# ML Readiness Audit: Participant and Trial Manifest

This notebook begins the implementation phase in the development plan.

It does not train a classifier. It creates a verified audit manifest for the current real labeled datasets before any windows, normalization or model code is written.

The existing EDA notebook remains unchanged.


## Audit goals

1. Confirm current participant and trial counts.
2. Confirm that each usable trial has the required sensor files.
3. Record labels, dataset source, participant identity, sampling rate and available confound variables.
4. Detect duplicate participant IDs and incomplete records.
5. Freeze the data contract before creating ML windows.


In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from features import felius, voisard

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print(f"Project root: {PROJECT_ROOT}")


Project root: C:\Users\frank\Documents\MR-ICT Review Paper


In [2]:
voisard_trials = voisard.list_trials().copy()
felius_trials = felius.list_trials().copy()

print("Voisard trials:", len(voisard_trials))
print("Felius complete sensor-triplet trials:", len(felius_trials))
print()
print("Voisard participants by cohort:")
print(voisard_trials.groupby("cohort")["subject"].nunique())
print()
print("Felius participants by label:")
print(felius_trials.groupby("label")["subject"].nunique())


Voisard trials: 488
Felius complete sensor-triplet trials: 377

Voisard participants by cohort:
cohort
CVA    49
HS     73
Name: subject, dtype: int64

Felius participants by label:
label
Healthy     34
Stroke     132
Name: subject, dtype: int64


In [3]:
def relative_path(path):
    return str(Path(path).resolve().relative_to(PROJECT_ROOT))


def build_voisard_manifest(trials):
    rows = []
    for row in trials.itertuples(index=False):
        meta = json.loads(Path(row.meta_path).read_text(encoding="utf-8"))
        files = sorted(relative_path(path) for path in Path(row.trial_dir).iterdir() if path.is_file())
        rows.append({
            "dataset": "Voisard",
            "dataset_id": "voisard_2025",
            "subject": meta.get("subject", row.subject),
            "trial_id": row.trial_id,
            "label": "stroke" if meta.get("pathologyKey") == "CVA" else "healthy",
            "source_label": meta.get("pathologyKey"),
            "age": meta.get("age"),
            "sex": meta.get("gender"),
            "walking_aid": np.nan,
            "gait_speed_m_s": np.nan,
            "sensor_configuration": meta.get("sensor"),
            "sampling_rate_hz": meta.get("freq"),
            "sampling_rate_source": "trial metadata",
            "clinical_deficit_side": meta.get("clinicalDeficitSide"),
            "evaluation_score_value": meta.get("evaluationScoreValue"),
            "raw_files": files,
            "trial_directory": relative_path(row.trial_dir),
        })
    return pd.DataFrame(rows)


def build_felius_manifest(trials):
    rows = []
    for row in trials.itertuples(index=False):
        files = sorted(relative_path(path) for path in row.paths.values())
        rows.append({
            "dataset": "Felius",
            "dataset_id": "felius_2024",
            "subject": row.subject,
            "trial_id": row.trial_key,
            "label": row.label.lower(),
            "source_label": row.label,
            "age": np.nan,
            "sex": np.nan,
            "walking_aid": np.nan,
            "gait_speed_m_s": np.nan,
            "sensor_configuration": "leftfoot,rightfoot,lowback",
            "sampling_rate_hz": 100.0,
            "sampling_rate_source": "project loader inference",
            "clinical_deficit_side": np.nan,
            "evaluation_score_value": np.nan,
            "raw_files": files,
            "trial_directory": relative_path(Path(files[0]).parent) if files else None,
        })
    return pd.DataFrame(rows)


manifest = pd.concat([
    build_voisard_manifest(voisard_trials),
    build_felius_manifest(felius_trials),
], ignore_index=True)

manifest.head(3)


,dataset,dataset_id,subject,trial_id,label,source_label,age,sex,walking_aid,gait_speed_m_s,sensor_configuration,sampling_rate_hz,sampling_rate_source,clinical_deficit_side,evaluation_score_value,raw_files,trial_directory
0,Voisard,voisard_2025,HS_1,HS_1_1,healthy,HS,18.0,M,NaN,NaN,MTw Awinda XSens,100.0,trial metadata,None,NaN,[data\raw\voisard_2025\data\healthy\HS\HS_1\HS...,data\raw\voisard_2025\data\healthy\HS\HS_1\HS_1_1
1,Voisard,voisard_2025,HS_1,HS_1_2,healthy,HS,18.0,M,NaN,NaN,MTw Awinda XSens,100.0,trial metadata,None,NaN,[data\raw\voisard_2025\data\healthy\HS\HS_1\HS...,data\raw\voisard_2025\data\healthy\HS\HS_1\HS_1_2
2,Voisard,voisard_2025,HS_1,HS_1_3,healthy,HS,18.0,M,NaN,NaN,MTw Awinda XSens,100.0,trial metadata,None,NaN,[data\raw\voisard_2025\data\healthy\HS\HS_1\HS...,data\raw\voisard_2025\data\healthy\HS\HS_1\HS_1_3


In [4]:
participant_manifest = (
    manifest.groupby(["dataset", "dataset_id", "subject"], as_index=False)
    .agg(
        label=("label", "first"),
        n_trials=("trial_id", "nunique"),
        age=("age", "first"),
        sex=("sex", "first"),
        sampling_rates_hz=("sampling_rate_hz", lambda x: tuple(sorted(set(x.dropna())))),
    )
)

print("Trial counts:")
print(manifest.groupby(["dataset", "label"]).size().rename("trials"))
print()
print("Participant counts:")
print(participant_manifest.groupby(["dataset", "label"]).size().rename("participants"))
print()
print("Duplicate subject IDs within a dataset:")
duplicates = participant_manifest[participant_manifest.duplicated(["dataset", "subject"], keep=False)]
print(duplicates if not duplicates.empty else "None")
print()
print("Subject IDs appearing across both datasets:")
voisard_ids = set(participant_manifest.loc[participant_manifest.dataset == "Voisard", "subject"])
felius_ids = set(participant_manifest.loc[participant_manifest.dataset == "Felius", "subject"])
cross_dataset_ids = voisard_ids & felius_ids
print(sorted(cross_dataset_ids) if cross_dataset_ids else "None")


Trial counts:
dataset  label  
Felius   healthy     59
         stroke     318
Voisard  healthy    360
         stroke     128
Name: trials, dtype: int64

Participant counts:
dataset  label  
Felius   healthy     34
         stroke     132
Voisard  healthy     73
         stroke      49
Name: participants, dtype: int64

Duplicate subject IDs within a dataset:
None

Subject IDs appearing across both datasets:
None


In [5]:
def existing_file_count(file_list):
    return sum((PROJECT_ROOT / path).exists() for path in file_list)


manifest["n_files"] = manifest["raw_files"].map(len)
manifest["n_existing_files"] = manifest["raw_files"].map(existing_file_count)
manifest["files_complete"] = manifest["n_files"] == manifest["n_existing_files"]

print("File completeness:")
print(manifest["files_complete"].value_counts(dropna=False))
print()
print("Sampling-rate availability:")
print(manifest.groupby(["dataset", "sampling_rate_source"])["sampling_rate_hz"].agg(["count", "min", "max"]))
print()
print("Confound-variable availability:")
availability = pd.DataFrame({
    "available_trials": manifest.notna().sum(),
    "total_trials": len(manifest),
})
availability["availability_fraction"] = availability["available_trials"] / availability["total_trials"]
print(availability.loc[["age", "sex", "walking_aid", "gait_speed_m_s", "clinical_deficit_side"]])


File completeness:
files_complete
True    865
Name: count, dtype: int64

Sampling-rate availability:
                                  count    min    max
dataset sampling_rate_source                         
Felius  project loader inference    377  100.0  100.0
Voisard trial metadata              488  100.0  100.0

Confound-variable availability:
                       available_trials  total_trials  availability_fraction
age                                 488           865               0.564162
sex                                 488           865               0.564162
walking_aid                           0           865               0.000000
gait_speed_m_s                        0           865               0.000000
clinical_deficit_side               128           865               0.147977


In [6]:
manifest_path = PROJECT_ROOT / "data" / "interim" / "ml_readiness_manifest.csv"
export = manifest.copy()
export["raw_files"] = export["raw_files"].map(lambda values: "|".join(values))
export.to_csv(manifest_path, index=False)
print(f"Wrote {manifest_path}")


Wrote C:\Users\frank\Documents\MR-ICT Review Paper\data\interim\ml_readiness_manifest.csv


## Gate before model code

Do not create training windows until these checks are reviewed:

1. Confirm the participant and trial counts above.
2. Confirm that every retained trial has the intended sensor files.
3. Decide the policy for Felius trials where stride detection fails. Raw-signal CNN windows may still be usable, but feature-based baselines need an explicit missing-value rule.
4. Decide how age, sex and gait speed will be used in confound analysis. They should not automatically become CNN inputs.
5. Build participant-level train, validation and test groups before window generation.

The next notebook should build the common walking-segment and fixed-window tensors only after this gate is accepted.


## 2026-09-02: Frozen NONAN healthy-specificity materialization

This records the predeclared, **healthy-only** NONAN GaitPrint examination set. It is separate from the three-person structural audit, the 80-person prospective enrichment partition, the Felius/Voisard development pool, and frozen RevalExo. It must not influence model fitting, normalization, source adaptation, calibration, threshold selection, or model selection.

Each publisher-verified archive was converted from mG to g; tri-axial data were anti-aliased from 200 Hz to 100 Hz using `resample_poly`; magnitudes were calculated in LB-proxy/LF/RF order; and non-overlapping five-second windows were generated. The raw representation is primary. A second representation repairs only interior runs of at most two vector-magnitude samples above 16 g, before resampling, as a predeclared isolated-artifact sensitivity check. The implementation is `scripts/materialize_nonan_staged_audit.py`.


In [7]:
from pathlib import Path
import json

NONAN_DIR = PROJECT_ROOT / 'data' / 'interim' / 'nonan_gaitprint'
download_manifest = json.loads((NONAN_DIR / 'download_manifest_frozen_healthy_specificity.json').read_text(encoding='utf-8'))
partitions = pd.read_csv(NONAN_DIR / 'participant_partitions.csv')
frozen = partitions.loc[partitions['partition'].eq('frozen_healthy_specificity')].copy()
metadata = pd.read_csv(NONAN_DIR / 'frozen_healthy_specificity_window_metadata.csv')
raw = np.load(NONAN_DIR / 'frozen_healthy_specificity_magnitude_raw.npy', mmap_mode='r')
repaired = np.load(NONAN_DIR / 'frozen_healthy_specificity_magnitude_isolated_spike_repaired.npy', mmap_mode='r')
materialization = json.loads((NONAN_DIR / 'frozen_healthy_specificity_materialization.json').read_text(encoding='utf-8'))

assert len(download_manifest) == 29 and all(item['status'] == 'downloaded_verified' for item in download_manifest)
assert raw.shape == repaired.shape == (23657, 500, 3)
assert np.isfinite(raw).all() and np.isfinite(repaired).all()
assert set(metadata['participant']) == set(frozen['participant_id'])
assert materialization['patched_samples'] == 167

cohort_by_participant = frozen.set_index('participant_id')['cohort']
coverage = (metadata.assign(cohort=metadata['participant'].map(cohort_by_participant))
            .groupby('cohort', sort=True)
            .agg(participants=('participant', 'nunique'), windows=('participant', 'size')))
print(f"Verified archives: {sum(item['status'] == 'downloaded_verified' for item in download_manifest)}/{len(download_manifest)}")
print(f"Frozen participants: {metadata['participant'].nunique()}/{len(frozen)}; all metadata participants are in the frozen partition: {set(metadata['participant']) == set(frozen['participant_id'])}")
print(f"Window arrays: raw={raw.shape}, repaired={repaired.shape}, finite={np.isfinite(raw).all() and np.isfinite(repaired).all()}")
print(f"Isolated samples repaired: {materialization['patched_samples']}; raw max={float(raw.max()):.3f} g; repaired max={float(repaired.max()):.3f} g")
for cohort, row in coverage.iterrows():
    print(f"{cohort}: {int(row['participants'])} participants, {int(row['windows'])} windows")


Verified archives: 29/29
Frozen participants: 29/29; all metadata participants are in the frozen partition: True
Window arrays: raw=(23657, 500, 3), repaired=(23657, 500, 3), finite=True
Isolated samples repaired: 167; raw max=68.012 g; repaired max=14.102 g
middle: 11 participants, 9072 windows
older: 9 participants, 7241 windows
young: 9 participants, 7344 windows


### Locked next action

The next analysis is a **single healthy-specificity evaluation** of the fixed canonical model on the raw data, with the repaired representation reported only as a predeclared sensitivity analysis. This is not a second validation set for tuning: NONAN will not be used to revise the architecture, normalizer, adapter, probability calibration, or operating threshold. Because the cohort is healthy-only, report false-positive rate/specificity and participant-level uncertainty; do not report AUROC.


## One-time frozen healthy-specificity evaluation

This cell scores only the already selected `full_expanded_inception_prototype_seed_42.pt` checkpoint. It loads the checkpoint's stored development-only normalization statistics and does not fit anything to NONAN. The probability decision rule is the fixed neutral `0.50` reference; it is not selected, adjusted, or calibrated on NONAN. Raw is primary and isolated-spike repair is a sensitivity analysis, both evaluated in this single locked run.


In [8]:
from scipy.stats import norm
import torch
from torch import nn

class FrozenBlock(nn.Module):
    def __init__(self, channels, out_channels=16):
        super().__init__()
        bottleneck = min(32, channels)
        self.b = nn.Conv1d(channels, bottleneck, 1, bias=False)
        self.br = nn.ModuleList([nn.Conv1d(bottleneck, out_channels, kernel, padding=kernel // 2, bias=False) for kernel in (7, 15, 25)])
        self.pool = nn.Conv1d(channels, out_channels, 1, bias=False)
        self.bn = nn.BatchNorm1d(out_channels * 4)
        self.res = nn.Conv1d(channels, out_channels * 4, 1, bias=False) if channels != out_channels * 4 else nn.Identity()
    def forward(self, signals):
        bottleneck = self.b(signals)
        branches = [branch(bottleneck) for branch in self.br]
        branches.append(self.pool(nn.functional.max_pool1d(signals, 3, 1, 1)))
        return nn.functional.gelu(self.bn(torch.cat(branches, 1)) + self.res(signals))

class FrozenInception(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(FrozenBlock(3), nn.MaxPool1d(2), FrozenBlock(64), nn.AdaptiveAvgPool1d(1))
        self.c = nn.Sequential(nn.Flatten(), nn.Dropout(0.30), nn.Linear(64, 1))
    def forward(self, signals):
        return self.c(self.f(signals)).squeeze(1)

def wilson_interval(successes, total, confidence=0.95):
    z = norm.ppf(1 - (1 - confidence) / 2)
    proportion = successes / total
    denominator = 1 + z ** 2 / total
    center = (proportion + z ** 2 / (2 * total)) / denominator
    radius = z * np.sqrt(proportion * (1 - proportion) / total + z ** 2 / (4 * total ** 2)) / denominator
    return center - radius, center + radius

checkpoint_path = PROJECT_ROOT / 'data' / 'processed' / 'full_expanded_inception_prototype_seed_42.pt'
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
assert checkpoint['strategy'] == 'full_expanded_felius_voisard_sint_source_class_balanced'
assert checkpoint['seed'] == 42
assert checkpoint['participants'] == 314 and checkpoint['windows'] == 22506

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FrozenInception().to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
mean = np.asarray(checkpoint['mean'], dtype='float32').reshape(1, 1, 3)
std = np.asarray(checkpoint['std'], dtype='float32').reshape(1, 1, 3)

def predict_frozen(windows, batch_size=512):
    probabilities = []
    with torch.no_grad():
        for start in range(0, len(windows), batch_size):
            batch = np.asarray(windows[start:start + batch_size], dtype='float32')
            tensor = torch.from_numpy(((batch - mean) / std).transpose(0, 2, 1).copy()).to(device)
            probabilities.append(torch.sigmoid(model(tensor)).cpu().numpy())
    return np.concatenate(probabilities)

results = []
for representation, windows in [('raw', raw), ('isolated_spike_repaired', repaired)]:
    window_probability = predict_frozen(windows)
    participant_prediction = metadata[['participant']].copy()
    participant_prediction['probability'] = window_probability
    participant_prediction = participant_prediction.groupby('participant', as_index=False)['probability'].mean()
    participant_prediction['representation'] = representation
    participant_prediction['predicted_stroke_at_0_50'] = participant_prediction['probability'].ge(0.50)
    false_positives = int(participant_prediction['predicted_stroke_at_0_50'].sum())
    specificity = 1 - false_positives / len(participant_prediction)
    ci_low, ci_high = wilson_interval(len(participant_prediction) - false_positives, len(participant_prediction))
    results.append({'representation': representation, 'participants': len(participant_prediction), 'false_positives_at_0_50': false_positives, 'false_positive_rate_at_0_50': false_positives / len(participant_prediction), 'specificity_at_0_50': specificity, 'specificity_wilson_95_ci_low': ci_low, 'specificity_wilson_95_ci_high': ci_high, 'median_probability': participant_prediction['probability'].median(), 'max_probability': participant_prediction['probability'].max()})
    participant_prediction.to_csv(NONAN_DIR / f'frozen_healthy_specificity_{representation}_participant_predictions.csv', index=False)

metrics = pd.DataFrame(results)
metrics.to_csv(NONAN_DIR / 'frozen_healthy_specificity_fixed_model_metrics.csv', index=False)
print(f'Device: {device}; checkpoint seed: {checkpoint["seed"]}; model fitting on NONAN: none')
print(metrics.round(4).to_string(index=False))


Device: cuda; checkpoint seed: 42; model fitting on NONAN: none
         representation  participants  false_positives_at_0_50  false_positive_rate_at_0_50  specificity_at_0_50  specificity_wilson_95_ci_low  specificity_wilson_95_ci_high  median_probability  max_probability
                    raw            29                        2                        0.069                0.931                        0.7804                         0.9809              0.0115           0.6544
isolated_spike_repaired            29                        2                        0.069                0.931                        0.7804                         0.9809              0.0115           0.6544


### Interpretation boundary

This result measures only transport of the fixed binary model to a protocol-consistent, healthy-only source. It cannot estimate discrimination, stroke sensitivity, calibration quality, or clinical deployment readiness. A poor healthy specificity result is an important domain-shift finding, but it does not authorize tuning the model on this frozen cohort.


## Descriptive review of frozen healthy false positives

This is a post-score observational audit, not a source-adaptation or tuning step. It uses the already saved fixed-model participant probabilities and raw magnitude windows. The summaries below cannot be used to alter the model, normalizer, threshold, calibration, or frozen-cohort membership.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
NONAN_DIR = PROJECT_ROOT / 'data' / 'interim' / 'nonan_gaitprint'
windows = np.load(NONAN_DIR / 'frozen_healthy_specificity_magnitude_raw.npy', mmap_mode='r')
window_metadata = pd.read_csv(NONAN_DIR / 'frozen_healthy_specificity_window_metadata.csv')
prediction = pd.read_csv(NONAN_DIR / 'frozen_healthy_specificity_raw_participant_predictions.csv')
partition = pd.read_csv(NONAN_DIR / 'participant_partitions.csv').query("partition == 'frozen_healthy_specificity'")
assert len(prediction) == 29 and set(prediction['participant']) == set(partition['participant_id'])

rows = []
for participant, indices in window_metadata.groupby('participant').groups.items():
    signal = np.asarray(windows[np.asarray(list(indices), dtype=int)], dtype='float32')
    values = signal.reshape(-1, 3)
    dynamics = (np.diff(signal, axis=1) * 100.0).reshape(-1, 3)  # g/s at 100 Hz
    row = {'participant': participant}
    for channel_index, channel in enumerate(('lb', 'lf', 'rf')):
        channel_values = values[:, channel_index]
        row[f'{channel}_mean_g'] = float(channel_values.mean())
        row[f'{channel}_std_g'] = float(channel_values.std())
        row[f'{channel}_p95_g'] = float(np.quantile(channel_values, 0.95))
        row[f'{channel}_dynamic_rms_g_s'] = float(np.sqrt(np.mean(dynamics[:, channel_index] ** 2)))
    rows.append(row)

profile = pd.DataFrame(rows).merge(prediction[['participant', 'probability', 'predicted_stroke_at_0_50']], on='participant').merge(partition[['participant_id', 'cohort', 'age_years', 'gender', 'mobility_flag_count']], left_on='participant', right_on='participant_id').drop(columns='participant_id')
feature_columns = [column for column in profile.columns if column.endswith(('_mean_g', '_std_g', '_p95_g', '_dynamic_rms_g_s'))]
for column in feature_columns:
    profile[f'{column}_percentile'] = profile[column].rank(pct=True) * 100.0
profile.to_csv(NONAN_DIR / 'frozen_healthy_specificity_descriptive_profile.csv', index=False)

false_positives = profile.loc[profile['predicted_stroke_at_0_50']].sort_values('probability', ascending=False)
percentile_columns = ['participant', 'cohort', 'age_years', 'gender', 'mobility_flag_count', 'probability'] + [f'{column}_percentile' for column in feature_columns]
print('False-positive participants and predeclared screening fields:')
print(false_positives[['participant', 'cohort', 'age_years', 'gender', 'mobility_flag_count', 'probability']].round(4).to_string(index=False))
print('\nTheir within-frozen-cohort signal-summary percentiles (not model features):')
print(false_positives[percentile_columns].round(1).to_string(index=False))
print('\nInterpretation: S043 is high-dynamics across LB/LF/RF; S131 is low on the same summaries. There is no shared amplitude or isolated-spike explanation in this 29-person cohort.')


False-positive participants and predeclared screening fields:
participant cohort  age_years gender  mobility_flag_count  probability
       S131  older         56 female                    0       0.6544
       S043 middle         54 female                    0       0.5299

Their within-frozen-cohort signal-summary percentiles (not model features):
participant cohort  age_years gender  mobility_flag_count  probability  lb_mean_g_percentile  lb_std_g_percentile  lb_p95_g_percentile  lb_dynamic_rms_g_s_percentile  lf_mean_g_percentile  lf_std_g_percentile  lf_p95_g_percentile  lf_dynamic_rms_g_s_percentile  rf_mean_g_percentile  rf_std_g_percentile  rf_p95_g_percentile  rf_dynamic_rms_g_s_percentile
       S131  older         56 female                    0          0.7                   3.4                  3.4                  6.9                            6.9                   3.4                 13.8                 17.2                           17.2                   3.4          

### Result and constraint

S043 (middle cohort, age 54) is at the extreme high end of lower-back and bilateral-foot dynamics, while S131 (older cohort, age 56) is near the low end on those same summaries. Both had zero recorded mobility-relevant screening flags. Thus the two false positives do not support a single clipping, scaling, age, or artifact correction. The correct next use of NONAN remains a separately predeclared source-balanced training sensitivity stream using the untouched 80-person candidate partition; the frozen 29 remain reporting-only.
